In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc qiskit-ibm-catalog
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

In [1]:
# This cell is hidden from users
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()
instance = service.active_account()["instance"]
backend_name = service.least_busy(operational=True, min_num_qubits=16).name

> **Note:** دوال Qiskit ميزة تجريبية متاحة فقط لمستخدمي خطط IBM Quantum&reg; Premium وFlex وOn-Prem (عبر IBM Quantum Platform API). هي في مرحلة إصدار معاينة وقابلة للتغيير.

## نظرة عامة
في الكيمياء الكمومية، تتمحور مسألة البنية الإلكترونية حول إيجاد حلول معادلة شرودنغر الإلكترونية — دوال الموجة الكمومية التي تصف سلوك إلكترونات النظام. هذه الدوال عبارة عن متجهات من السعات المركبة، حيث تقابل كل سعة مساهمة إحدى التهيئات الإلكترونية الممكنة.

الحالة الأرضية هي دالة الموجة ذات أدنى طاقة في النظام، وتحتل مكانة خاصة في دراسة الأنظمة الجزيئية. يأخذ النهج الأدق لحساب الحالة الأرضية بعين الاعتبار جميع التهيئات الإلكترونية الممكنة، إلا أن ذلك يصبح غير قابل للتطبيق على الأنظمة الأكبر إذ يتزايد عدد التهيئات بشكل أسي مع حجم النظام.

يُعدّ الـ HI-VQE (Handover Iterative Variational Quantum Eigensolver) منهجًا هجينًا مبتكرًا بين الحوسبة الكمومية والكلاسيكية، يهدف إلى تقدير دقيق للحالة الأرضية للأنظمة الجزيئية. يدمج المعالجات الكمومية مع الحوسبة الكلاسيكية، مستعينًا بالمعالجات الكمومية لاستكشاف التهيئات الإلكترونية المرشحة بكفاءة، ثم يحسب دالة الموجة الناتجة على الحواسيب الكلاسيكية. من خلال توليد دوال موجة مضغوطة وذات دقة كيميائية عالية، يُعزّز HI-VQE البحث والاكتشاف في كيمياء الكم وعلم المواد.

![صورة توضح نظرة عامة على خوارزمية HI-VQE من Qunova](../docs/images/guides/qunova-chemistry/overview.svg)

يُقلّل HI-VQE من التعقيد الحسابي لمسألة البنية الإلكترونية من خلال تقدير الحالة الأرضية بدقة عالية وكفاءة. يركّز على مجموعة فرعية مختارة بعناية من أبرز التهيئات الإلكترونية، محقّقًا توازنًا مثاليًا بين الدقة والكفاءة.

يجمع HI-VQE بين إمكانيات الحواسيب الكلاسيكية والكمومية، ويُحسّن تقدير دالة الموجة الحالية تدريجيًا وبصورة متكررة. تُسهم تقنياته الفريدة لبناء الفضاء الجزئي في رفع كفاءة اختيار التهيئات، مما يمنح المستخدمين تحكّمًا حسابيًا أكبر ودقة محسّنة في محاكاة كيمياء الكم.

إن كنت تودّ التعمق في فهم الخوارزمية، يمكنك [قراءة الورقة البحثية المرتبطة](https://arxiv.org/abs/2503.06292).
## الوصف
يتزايد عدد التهيئات الإلكترونية لنظام جزيئي بشكل أسي مع حجمه. غير أنه في حالات إلكترونية معينة، كالحالة الأرضية، يشيع أن نسبة صغيرة فقط من التهيئات تُسهم بشكل ملحوظ في طاقة الحالة. تستغل أساليب التفاعل التشكيلي المنتقى (SCI) هذه الندرة لتقليل التكاليف الحسابية عبر تحديد التهيئات الأكثر صلة والتركيز عليها. تُسمّى هذه المجموعة الفرعية من التهيئات بالفضاء الجزئي.

يستثمر HI-VQE الكفاءة الجوهرية للحواسيب الكمومية في تمثيل الأنظمة الجزيئية للمساعدة في البحث عن الفضاء الجزئي. يدمج الروتينات الكلاسيكية والكمومية لحل مسألة البنية الإلكترونية بدقة عالية. على خلاف أساليب SCI الكمومية الموجودة، يجمع HI-VQE بين التدريب التغايري والبناء التكراري للفضاء الجزئي وفرز التهيئات قبل التقطير، لتعزيز الكفاءة عبر تقليل القياسات الكمومية والتكرارات وتكاليف التقطير الكلاسيكية. وبالتالي يمكن تطبيق HI-VQE على أنظمة جزيئية أكبر تحتاج إلى مزيد من qubits، مع تقليل تكلفة حل مسألة بحجم معين بالدقة ذاتها.

![صورة توضح وصفًا تفصيليًا لكل خطوة في خوارزمية HI-VQE من Qunova.](../docs/images/guides/qunova-chemistry/description.avif)

لحساب الحالة الأرضية لنظام ما، يبدأ HI-VQE بالاستعانة بحزمة الكيمياء الكلاسيكية PySCF لتوليد تمثيل جزيئي من المدخلات التي يوفّرها المستخدم، كالهندسة الجزيئية وبيانات أخرى عن الجزيء. ثم يدخل في حلقة تحسين هجينة كمومية-كلاسيكية، يُحسّن فيها تدريجيًا فضاءً جزئيًا لتمثيل الحالة الأرضية بشكل أمثل مع تقليل عدد التهيئات المضمّنة. تستمر الحلقة حتى تتحقق معايير التقارب كحجم الفضاء الجزئي أو استقرار الطاقة، ثم يُخرج الخوارزم دالة موجة الحالة الأرضية المحسوبة وطاقتها. يمكن استخدام هذه النتائج لبناء أسطح طاقة الوضع الدقيقة وإجراء مزيد من التحليل للنظام.

تتمحور حلقة التحسين حول ضبط معاملات دائرة كمومية لتوليد فضاء جزئي عالي الجودة. يوفّر HI-VQE ثلاثة خيارات للدائرة الكمومية: [`excitation_preserving`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.excitation_preserving) و[efficient_su2](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.efficient_su2) و[LUCJ](https://qiskit-community.github.io/ffsim/explanations/lucj.html). يُهيّأ التحسين قريبًا من حالة مرجعية Hartree-Fock لملاءمتها العامة. تُنفَّذ الدائرة بعد ذلك على جهاز كمومي وتُؤخذ عيّنات من التهيئات من الحالة الكمومية الناتجة وتُعاد كسلاسل ثنائية. نتيجةً لضوضاء الجهاز الكمومي، قد تكون بعض التهيئات المُعيَّنة غير صالحة فيزيائيًا، إذ لا تحفظ عدد الإلكترونات أو الدوران. يتعامل HI-VQE مع هذا باستخدام عملية استرداد التهيئة من حزمة [qiskit-addon-sqd](/guides/qiskit-addons-sqd#sample-based-quantum-diagonalization-sqd-overview)، ليتمكن المستخدمون إما من تصحيح التهيئات غير الصالحة أو تجاهلها.

تخضع التهيئات الصالحة بعد ذلك لخطوة فرز اختيارية لإزالة تلك المتوقع أن تُسهم بالحد الأدنى. يُقلّل ذلك من بُعد الفضاء الجزئي وبالتالي يخفض تكلفة خطوة التقطير. في حال تفعيل الفرز، يُبنى Hamiltonian فضاء جزئي أوّلي من التهيئات الصالحة ويُجرى تقطير بمعايير إنهاء فضفاضة جدًا. رغم أن دقة السعات الناتجة لكل تهيئة منخفضة، إلا أنها فعّالة في التنبؤ بالتهيئات التي ينبغي استبعادها من الفضاء الجزئي في هذه الجولة، وتُحسب بسرعة.

تُضاف التهيئات المختارة إلى الفضاء الجزئي ويُسقَط Hamiltonian النظام في هذا الفضاء. يتحدث الفضاء الجزئي تكراريًا محافظًا على أبرز التهيئات عبر الجولات. يتميز هذا النهج عن البدائل بأن الدائرة الكمومية لا تحتاج إلى تقريب الحالة الأرضية الكاملة في كل خطوة.

بعد ذلك، يُقطَّر Hamiltonian الفضاء الجزئي كلاسيكيًا للحصول على أدنى قيمة ذاتية ومتجهها الذاتي المقابل، وهما تقريب للحالة الأرضية وطاقتها. مع تحسّن جودة الفضاء الجزئي عبر الجولات، يقترب حساب الحالة الأرضية من الحالة الحقيقية. يمكن إجراء خطوة فرز إضافية في هذه المرحلة لإزالة التهيئات ذات المساهمة غير الجوهرية في الحالة الأرضية المحسوبة. تضمن هذه الخطوة أن الفضاء الجزئي الذي يُنقل للجولة التالية يكون بالحجم الأمثل. ويُقيَّم ذلك بناءً على السعات التي تُعيدها عملية التقطير، إذ تمثّل أهمية مساهمة كل تهيئة في الحالة الأرضية المحسوبة.

ثم يُجرى فحص تقارب لتحديد ما إذا كانت التكرارات الإضافية ستحسّن النتائج. إن كان الأمر كذلك، تُجرى خطوة توسيع كلاسيكية اختيارية، تُحدَّث معاملات الدائرة الكمومية لمزيد من تقليل الطاقة المحسوبة، وتتكرر العملية. تُولّد خطوة التوسيع الكلاسيكية تهيئات إضافية للفضاء الجزئي تكمّل ما أُخذ عيّنات منه من الجهاز الكمومي. تُحدّد أولًا التهيئة ذات أكبر سعة في نتائج التقطير، ثم تُولّد تهيئات جديدة بإثارات أحادية وثنائية من التهيئة المحدّدة، وتُضاف العدد المطلوب من هذه التهيئات إلى الفضاء الجزئي.

بمجرد التحقق من تقارب الجولات، يُعيد HI-VQE الحالة الأرضية المحسوبة (على شكل الحالات في الفضاء الجزئي وسعاتها في دالة الموجة)، وطاقتها، ومقياس تباين الطاقة الذي يشير إلى ما إذا كانت الحالة المحسوبة تُشكّل حالة ذاتية لـ Hamiltonian النظام.

يستطيع المستخدمون تحديد الدائرة الكمومية المستخدمة وعدد اللقطات لكل دائرة، إضافةً إلى التحكم في حجم الفضاء الجزئي أو تفعيل التوليد الكلاسيكي لتهيئات إضافية لدعم تلك المولّدة كميًا. بهذه الطريقة، يمكن للمستخدمين تكييف سلوك HI-VQE لتناسب تطبيقاتهم المطلوبة.
## البدء
أولًا، [اطلب الوصول إلى الدالة](https://forms.office.com/r/zN3hvMTqJ1).
ثم قم بالمصادقة باستخدام [مفتاح IBM Quantum&reg; API](http://quantum.cloud.ibm.com/) الخاص بك، وبافتراض أنك [حفظت حسابك](/guides/functions#install-qiskit-functions-catalog-client) في بيئتك المحلية، اختر دالة Qiskit كما يلي:

In [ ]:
import reprlib
from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

function = catalog.load("qunova/hivqe-chemistry")

## المدخلات
انظر الجدول التالي لجميع معاملات المدخلات التي تقبلها الدالة. يُفصّل قسما [خيارات الجزيء](#molecule-options) و[خيارات HI-VQE](#hi-vqe-options) هذه الوسيطات بمزيد من التفاصيل.
| الاسم                   | النوع                                                           | الوصف                                                                                                                                                                                                                                                                                                 | مطلوب | الافتراضي                                                                  | مثال                                                                                   |
|------------------------|----------------------------------------------------------------|-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|----------|--------------------------------------------------------------------------|-------------------------------------------------------------------------------------------|
| geometry               | Union[List[List[Union[str, Tuple[float, float, float]]]], str] | يمكن أن تكون سلسلة نصية أو قوائم منظّمة تحتوي على أزواج من الذرة والإحداثيات. إذا كانت سلسلة نصية، فيجب أن تكون هندسة جزيئية بصيغة إحداثيات ديكارتية. إذا كانت قائمة، فيجب أن تكون قائمة من القوائم تحتوي كل منها على سلسلة ذرة وزوج إحداثيات. | نعم      | N/A                                                                      | `[['O', (0, 0, 0)], ['H', (0, 1, 0)], ['H', (0, 0, 1)]]` أو `"O 0 0 0; H 0 1 0; H 0 0 1"` |
| backend\_name          | str                                                            | اسم الـ Backend لتنفيذ الاستعلام عليه.                                                                                                                                                                                                                                                                      | نعم      | N/A                                                                      | `ibm_fez`                                                                                 |
| max\_states            | int                                                            | أقصى بُعد للفضاء الجزئي في التقطير. سيُستخدم عدد أقل من الحالات إذا لم يكن الرقم مربعًا تامًا.                                                                                                                                                                                                                                                                    | نعم      | N/A                                                                      | `100`                                                                                     |
| max\_expansion\_states | int                                                            | أقصى عدد لحالات CI المولّدة كلاسيكيًا لتضمينها في كل جولة.                                                                                                                                                                                                                                     | نعم      | N/A                                                                      | `10`                                                                                      |
| molecule_options                | dict                                                           | خيارات متعلقة بالجزيء المستخدم كمدخل لـ HI-VQE. انظر قسم [خيارات الجزيء](#molecule-options) لمزيد من التفاصيل.                                                                                                                                                                                                                                                | لا       | انظر قسم [خيارات الجزيء](#molecule-options) للتفاصيل.                                 | `{"basis": "sto3g", "unit": "angstrom" }`                               |
| hivqe_options                | dict                                                           | خيارات تتحكم في سلوك خوارزمية HI-VQE. انظر قسم [خيارات HI-VQE](#hi-vqe-options) لمزيد من التفاصيل.                                                                                                                                                                                                                                                | لا       | انظر قسم [خيارات HI-VQE](#hi-vqe-options) للتفاصيل.                                 | `{"shots": 10_000, "max_iter": 10 }`                               |
### خيارات الجزيء
يوضّح الجدول التالي جميع المفاتيح والقيم التي يمكن ضبطها في قاموس `molecule_options`، إضافةً إلى أنواع بياناتها وقيمها الافتراضية. جميع المفاتيح اختيارية.

| المفتاح                               | نوع القيمة                          | القيمة الافتراضية                    | النطاق الصالح                                                                                                                                                    | الشرح                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                            |
|:----------------------------------|:-----------------------------------:|:--------------------------------:|:---------------------------------------------------------------------------------------------------------------------------------------------------------------|:----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| `"charge"`                        | `int`                               | `0`                              | متنوع                                                                                                                                                        | عدد صحيح يحدد الشحنة الكلية للنظام الجزيئي. القيمة الافتراضية هي 0، غير أنه يمكن أن يكون أي عدد صحيح.                                                                                                                                                                                                                                                                                                                                                                                                                              |
| `"basis"`                         | `str`                               | `'sto-3g'`                       | متنوع                                                                                                                                                        | سلسلة نصية تحدد نوع الأساس؛ تُمرَّر إلى pyscf. (مثال: `"sto-3g"`, `"3-21g"`, `"6-31g"`, `"cc-pvdz"`)                                                                                                                                                                                                                                                                                                                                                                                                                      |
| `"active_orbitals"`               | `List[int]`                         | كل مؤشرات المدارات.             | مؤشرات المدارات الفضائية الصالحة للمسألة.                                                                                                             | قائمة بمؤشرات المدارات النشطة في النطاق [0, n) حيث n هو عدد الـ qubits المستخدمة في المسألة. إذا تم تحديده، يجب تحديد وسيطة frozen_orbitals أيضًا.                                                                                                                                                                                                                                                                                                                                            |
| `"frozen_orbitals"`               | `List[int]`                         | لا مؤشرات.                      | مؤشرات المدارات الفضائية الصالحة للمسألة، باستثناء المدارات النشطة.                                                                                  | قائمة بمؤشرات المدارات المجمّدة في النطاق ذاته كـ active_orbitals. إذا تم تحديده، يجب تحديد active_orbitals أيضًا. لاحظ أنه يجب تجميد المدارات المشغولة فقط إذ ينخفض عدد الإلكترونات النشطة بمقدار 2 لكل مدار مشغول مجمّد.                                                                                                                                                                                                                                                        |
| `"orbital_coeffs"`                | `List[List[float]]`                 | المدارات الجزيئية Hartree-Fock. | متنوع.                                                                                                                                                       | معاملات المدارات الفضائية المستخدمة في حساب تكاملات الطرد الإلكتروني للنظام. بعض الأمثلة الصالحة هي مدارات Hartree-Fock الجزيئية والمدارات الطبيعية ومدارات AVAS.                                                                                                                                                                                                                                                                                                                                                                   |
| `"symmetry"`                      | `Union[str, bool]`                  | `False`                          | `True` أو `False`                                                                                                                                              | يُستخدم لاستدعاء تماثل المجموعة النقطية في الحسابات الجزيئية الأولية لبناء أساس المدارات المتكيّفة مع التماثل. تُستخدم هذه المدارات كدوال أساسية لحسابات SCF التالية. القيمة الافتراضية False؛ إذا ضُبطت على True، سيُستدعى تلقائيًا وستُكتشف مجموعات النقاط التعسفية وتُستخدم. إذا حُدّد تماثل معين، مثلًا symmetry = "Dooh"، فسيُطلق خطأ إذا لم تكن الهندسة الجزيئية خاضعة لهذا التماثل المطلوب. |
| `"symmetry_subgroup"`             | `Optional[str]`                     | `None`                           | انظر [توثيق pyscf](https://pyscf.org/pyscf_api_docs/pyscf.gto.html#pyscf.gto.mole.MoleBase.build).                                                      | يمكن استخدامه لتوليد مجموعة فرعية من التماثل المكتشف. لا تأثير له عند تحديد التماثل باستخدام وسيطة الكلمة المفتاحية symmetry.                                                                                                                                                                                                                                                                                                                                                                                                                         |
| `"unit"`                          | `str`                               | `"angstrom"`                     | انظر [توثيق pyscf](https://pyscf.org/pyscf_api_docs/pyscf.gto.html#pyscf.gto.mole.MoleBase.build).                                                      | يحدد وحدة القياس المستخدمة للإحداثيات والمسافات الذرية. الافتراضي هو وحدة أنغستروم.                                                                                                                                                                                                                                                                                                                                                                                                                                       |
| `"nucmod"`                        | `Optional[Union[dict, str]]`        | `None`                           | انظر [توثيق pyscf](https://pyscf.org/pyscf_api_docs/pyscf.gto.html#pyscf.gto.mole.MoleBase.build).                                                      | يحدد النموذج النووي المستخدم. افتراضيًا يستخدم النموذج النووي النقطي، والقيم الأخرى تتيح النموذج النووي الغاوسي. إذا أُعطيت دالة، ستُستخدم مع النموذج النووي الغاوسي لتوليد قيمة توزيع الشحنة النووية 'zeta'.                                                                                                                                                                                                                                                                   |
| `"pseudo"`                        | `Optional[Union[dict, str]]`        | `None`                           | انظر [توثيق pyscf](https://pyscf.org/pyscf_api_docs/pyscf.gto.html#pyscf.gto.mole.MoleBase.build).                                                      | يحدد الكمون الزائف للذرات في الجزيء. القيمة الافتراضية None تشير إلى عدم تطبيق أي كمونات زائفة وأن جميع الإلكترونات مضمّنة صراحةً في الحسابات.                                                                                                                                                                                                                                                                                                                                                                |
| `"cart"`                          | `bool`                              | `False`                          | انظر [توثيق pyscf](https://pyscf.org/pyscf_api_docs/pyscf.gto.html#pyscf.gto.mole.MoleBase.build).                                                      | يحدد ما إذا كان يجب استخدام GTO ديكارتية كدوال أساسية للزخم الزاوي في الحساب. القيمة الافتراضية False ستستخدم GTO كروية.                                                                                                                                                                                                                                                                                                                                                                               |
| `"magmom"`                        | `Optional[List[Union[int, float]]]` | `None`                           | انظر [توثيق pyscf](https://pyscf.org/pyscf_api_docs/pyscf.gto.html#pyscf.gto.mole.MoleBase.build).                                                      | يضبط العزم المغناطيسي الدوراني الخطي لكل ذرة. افتراضيًا هو None ويُهيّأ كل ذرة بدوران صفري.                                                                                                                                                                                                                                                                                                                                                                                                                                         |
| `"avas_aolabels"`                 | `Optional[List[str]]`               | `None`                           | مثال: ["H 1s", "O 2p"] لـ H2O                                                                                                                                                             | يحدد المدارات الذرية لتضمينها في مخطط AVAS. انظر [توثيق AVAS](https://arxiv.org/abs/1701.07862).                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                         |
| `"avas_threshold"`                | `float`                             | `0.2`                            |  بين 0.0 و2.0                                                                                                                                                      |  يحدد قيمة القطع المستخدمة لتحديد المدارات الذرية (AOs) التي تُحتفظ في الفضاء النشط.                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                      |
| `"noons_level"`                   | `Optional[str]`                     | `None`                           | `"mp2"` أو `"ccsd"`                                                                                                                                            | يحدد النهج النظري لتحضير المدارات الطبيعية واختيار المدارات النشطة استنادًا إلى مخطط أعداد إشغال المدارات الطبيعية (NOONs). انظر [توثيق NOONs](https://doi.org/10.1063/5.0042147). يجب توفير مؤشرات المدارات النشطة والمجمّدة للتحكم في عدد المدارات (وعدد الـ qubits).                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                             |
### خيارات HI-VQE
يوضّح الجدول التالي جميع المفاتيح والقيم التي يمكن ضبطها في قاموس `hivqe_options`، إضافةً إلى أنواع بياناتها وقيمها الافتراضية. جميع المفاتيح اختيارية.

| المفتاح                               | نوع القيمة                          | القيمة الافتراضية                    | النطاق الصالح                                                                                                                                                    | الشرح                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                            |
|:----------------------------------|:-----------------------------------:|:--------------------------------:|:---------------------------------------------------------------------------------------------------------------------------------------------------------------|:----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| `"shots"`                         | `int`                               | `1_000`                          | بين 1 و10,000.                                                                                                                                          | عدد اللقطات لاستخدامها على الجهاز الكمومي لكل جولة.                                                                                                                                                                                                                                                                                                                                                                                                                                                             |
| `"max_iter"`                      | `int`                               | `25`                             | بين 1 و50.                                                                                                                                              | الحد الأقصى لعدد الجولات لتحسين الـ ansatz. قد يستخدم الخوارزم عددًا أقل من الجولات إذا تحقق التقارب مبكرًا.                                                                                                                                                                                                                                                                                                                                                                                                                                 |
| `"initial_basis_states"`          | `List[str]`                         | حالة Hartree-Fock.          | سلاسل ثنائية يتطابق عدد بتاتها مع عدد الـ qubits المطلوبة للمسألة.                                                                 | يمكن استخدامه لإعادة تشغيل الخوارزم بحالات كلاسيكية من نتيجة سابقة.                                                                                                                                                                                                                                                                                                                                                                                                                                                      |
| `"ansatz"`                        | `str`                               | `"epa"`                          | `"epa"` أو `"hea"` أو `"lucj"`                                                                                                                                  | يحدد الـ ansatz الكمومي المُحسَّن لتوليد حالات جديدة. `"epa"` يختار ansatz حافظ الإثارة. `"hea"` يختار ansatz كفء مع الأجهزة. `"lucj"` يختار ansatz مجموعة Jastrow الموحّدة المحلية.                                                                                                                                                                                                                                                                                                                       |
| `"convergence_count"`             | `int`                               | `3`                              | 2 على الأقل.                                                                                                                                                    | عدد الجولات بدون تغيير ملحوظ في الطاقة المحسوبة التي يجب أن تمر قبل اعتبار الخوارزم متقاربًا.                                                                                                                                                                                                                                                                                                                                                                                                                                         |
| `"convergence_abstol"`            | `float`                             | `1e-4`                           | أكبر من 0 وأقل من أو يساوي 1.                                                                                                                                     | مقدار التغيير في الطاقة المحسوبة الذي يُعدّ ملحوظًا لأغراض فحوصات التقارب.                                                                                                                                                                                                                                                                                                                                                                                                                                       |
| `"reset_convergence_count"`       | `bool`                              | `True`                           | `True` أو `False`                                                                                                                                              | إذا كانت `True`، يجب أن تحدث جولات `convergence_count` بدون تغيير ملحوظ يقطعها للتأهل كتقارب. إذا كانت `False`، فسيتوقف الخوارزم بعد `convergence_count` إذا حدثت تغييرات غير ملحوظة في أي جولات خلال عملية التحسين.                                                                                                                                                                 |
| `"configuration_recovery"`        | `bool`                              | `True`                           | `True` أو `False`.                                                                                                                                             | ما إذا كان يجب استخدام استرداد التهيئة من حزمة `qiskit-addon-sqd`. إذا كانت True، تُصحَّح الحالات غير الصالحة المأخوذة عيّنات من الجهاز الكمومي كلاسيكيًا. إذا كانت False، تُتجاهل.                                                                                                                                                                                                                                                                                                                                      |
| `"ansatz_entanglement"`           | `str`                               | `"circular"`                     | أي من `"linear"` أو `"reverse_linear"` أو `"pairwise"` أو `"circular"` أو `"full"` أو `"sca"`. عند استخدام الـ `"lucj"` ansatz، يتوفر أيضًا `"lucj_default"`. | يحدد مخطط التشابك الذي يجب استخدامه داخل الدائرة الكمومية، وفق اتفاقيات Qiskit الشائعة و[اتفاقيات ffsim للـ LUCJ ansatz](https://qiskit-community.github.io/ffsim/how-to-guides/simulate-lucj.html).                                                                                                                                                                                       |
| `"ansatz_reps"`                   | `int`                               | `2`                              | أكبر من 0.                                                                                                                                                | عدد تكرارات كل طبقة في الدائرة الكمومية.                                                                                                                                                                                                                                                                                                                                                                                                                                                         |
| `"amplitude_screening_tolerance"` | `Union[float,int]`                  | `0`                              | 0 على الأقل، وأقل من 1.                                                                                                                                   | الحد المتسامح لتقرير أي حالات يجب فرزها من الفضاء الجزئي بعد التقطير. يحدد عتبة الإدراج لحالات الفضاء الجزئي بناءً على سعاتها المحسوبة.                                                                                                                                                                                                                                                                                                                                                  |
| `"overlap_screening_tolerance"`   | `float`                             | `1e-2`                           | بين `1e-4` و`1e-1`، شاملًا.                                                                                                                          | الحد المتسامح للتنبؤ بأي حالات يجب فرزها من الفضاء الجزئي قبل التقطير. يتحكم في دقة السعات المتنبأ بها لكل حالة، مع نتيجة قيمة أصغر في تنبؤات أكثر دقة.                                                                                                                                                                                                                                                                                                                                                            |
## المخرجات
تُعيد الدالة قاموسًا بأربعة مفاتيح وقيم. المفاتيح والقيم موثّقة في الجدول التالي:

| المفتاح             | نوع القيمة    | الشرح                                                                                                               |
|:----------------|---------------|---------------------------------------------------------------------------------------------------------------------------|
| `"energy"`      | `float`       | طاقة الحالة الأرضية التقريبية للجزيء.                                                                      |
| `"states"`      | `List[str]`   | المحدِّدات المختارة التي تشكّل الفضاء الجزئي المستخدم لحل الطاقة. بصيغة ألفا-بيتا المتناوبة. |
| `"eigenvector"` | `List[float]` | المتجه الذاتي المقابل للحالة الأرضية للفضاء الجزئي المكوّن من `"states"`.                                 |
| `"energy_variance"` | `float` | تباين طاقة الحالة الأرضية للفضاء الجزئي المكوّن من `"states"`، يُشير إلى جودة الحل. هذه القيمة غير سلبية والقيمة الأقل تعني أن الحالة الأرضية للفضاء الجزئي تقترب أكثر من الحالة الذاتية لـ Hamiltonian النظام. |
| `"energy_history"` | `List[float]` | الطاقات المحسوبة في كل جولة خلال عملية التحسين الهجين، بالترتيب الذي حُسبت فيه. تُحسب طاقتان لكل جولة كجزء من عملية التحسين SPSA. |
## مثال
المثال الأول يوضح كيفية حساب طاقة الحالة الأرضية لجزيء NH3 باستخدام خوارزمية HI-VQE.
#### تحديد الهندسة الجزيئية والخيارات
تُوفَّر الهندسة الجزيئية لـ NH3 بإحداثيات ديكارتية مفصولة بـ ";" لكل ذرة.

In [3]:
# Define the molecule geometry
geometry = """
N         -0.85188       -0.02741        0.03141;
H          0.16545        0.00593       -0.01648;
H         -1.16348       -0.39357       -0.86702;
H         -1.16348        0.94228        0.06281;
"""

يمكن تعريف خيارات إضافية وتوفيرها للنظام الجزيئي بصيغة القاموس التالية.

In [4]:
# Configure some options for the job.
molecule_options = {"basis": "sto3g"}
hivqe_options = {"shots": 100, "max_iter": 20}

نفّذ الدالة مع مدخلات الهندسة والخيارات.

In [5]:
# Run HI-VQE
job = function.run(
    geometry=geometry,
    # `backend_name` is the name of a backend with at least 16 qubits, for example, "ibm_marrakesh".
    backend_name=backend_name,
    max_states=2000,
    max_expansion_states=10,
    molecule_options=molecule_options,
    hivqe_options=hivqe_options,
)

من المفيد طباعة معرّف مهمة الدالة حتى يمكن تقديمه في طلبات الدعم إذا حدث خطأ ما.

In [6]:
print("Job ID:", job.job_id)

Job ID: 128ee399-7cfc-4be2-91e9-c4abd22b97c7


This example then utilizes 16 qubits with 8 orbitals of sto3g basis for an NH3 molecule.

Check your Qiskit Function workload's [status](/docs/guides/functions#check-job-status) or return [results](/docs/guides/functions#retrieve-results) as follows:

In [7]:
print(job.status())

QUEUED


يستخدم هذا المثال 16 qubit مع 8 مدارات من أساس sto3g لجزيء NH3.
تحقق من [الحالة](/guides/functions#check-job-status) أو استرجع [النتائج](/guides/functions#retrieve-results) لعمل Qiskit Function الخاص بك على النحو التالي:

In [8]:
result = job.result()

# Output can be long, so we display a shortened representation
shortened_result = reprlib.repr(result)
print(shortened_result)

{'eigenvector': [0.9824200343205695, 0.009520846167419264, 6.365368845740147e-08, 3.6832123006425615e-07, 0.0012869941182066654, 2.3403891050875465e-05, ...], 'energy': -55.521146537970466, 'energy_history': [-55.52091922342852, -55.52113695367561, -55.521146537970466, -55.52114653864798, -55.521146537970466, -55.521146537970466, ...], 'energy_variance': 9.788555455342562e-12, ...}


To access the ground state energy, use the "energy" key. The "eigenvector" key provides the CI coefficients with corresponding bitstring notation of electron configuration stored with "states" of the results.

In [10]:
fci_energy = -55.521148034704126  # the exact energy using FCI method
hivqe_energy = result["energy"]
print(
    f"|Exact Energy - HI-VQE Energy|: {abs(fci_energy - hivqe_energy) * 1000} mHa"
)
print(f"Sampled Number of States: {len(result['states'])}")

|Exact Energy - HI-VQE Energy|: 0.0014967336596782843 mHa
Sampled Number of States: 1936


بعد اكتمال المهمة، يمكن الحصول على النتائج باستخدام نسخة `result()`.

In [11]:
# This cell is hidden from users
backend_name = service.least_busy(operational=True, min_num_qubits=38).name

In [12]:
# Define Li2S geometries
Li2S_geoms = {
    "Li2S_1.51": "S        -1.239044    0.671232   -0.030374;Li       -1.506327    0.432403   -1.498949;Li       -0.899996    0.973348    1.826768;",
    "Li2S_2.40": "S        -1.741432    0.680397    0.346702;Li       -0.529307    0.488006   -1.729343;Li       -1.284307    0.989409    2.177209;",
    "Li2S_3.80": "S        -2.707255    0.674298    0.909161;Li        0.079218    0.552012   -1.671656;Li       -0.927010    0.931502    1.557063;",
}

# Configure some options for the job.
molecule_options = {
    "basis": "sto3g",
}
hivqe_options = {
    "shots": 100,
    "max_iter": 20,
}

results = []
for geom in ["Li2S_1.51", "Li2S_2.40", "Li2S_3.80"]:
    # Run HI-VQE
    job = function.run(
        geometry=Li2S_geoms[geom],
        backend_name=backend_name,  # can use any device with at least 38 qubits
        max_states=2000,
        max_expansion_states=10,
        molecule_options=molecule_options,
        hivqe_options=hivqe_options,
    )
    results.append(job.result())

للوصول إلى طاقة الحالة الأرضية، استخدم مفتاح "energy". يوفّر مفتاح "eigenvector" معاملات CI مع تدوين السلسلة الثنائية المقابلة للتهيئة الإلكترونية المخزّنة في "states" من النتائج.

In [13]:
job = function.run(
    geometry="invalid-geometry",  # This will cause an error
    backend_name=backend_name,
    max_states=2000,
    max_expansion_states=15,
    molecule_options=molecule_options,
    hivqe_options=hivqe_options,
)

job.result()

QiskitServerlessException: {"message": "An unexpected error occurred during job execution. Please make sure that your inputs are valid. If you are still experiencing problems, you can contact the Qunova Computing support service at qiskit.support@qunovacomputing.com and provide the Function job ID of this job for more assistance.", "status": "failure"}

In [14]:
job.status()

'ERROR'

المخرجات:

|Exact Energy - HI-VQE Energy|: 0.077237504 mHa
Sampled Number of States: 1936
## الأداء
يعرض هذا القسم حسابات المعايير التوضيحية لـ HI-VQE مع حالة بـ 24 qubit لجزيء Li2S، وحالة بـ 40 qubit لجزيء N2، وحالة بـ 44 qubit لنظام FeP-NO.
#### منحنى سطح طاقة الوضع للتفكك لجزيء Li2S بـ 24 qubit
يُعرض منحنى PES مع مرجع FCI والتخمين الأوّلي من RHF، إضافةً إلى خطأ الطاقة من مرجع FCI.

![صورة توضح أن HI-VQE ينتج حلولًا ضمن الدقة الكيميائية لمنحنى PES المرجعي الكلاسيكي لنظام Li2S.](../docs/images/guides/qunova-chemistry/Li2S_PES.avif)

أُجريت الحسابات بالهندسات والخيارات التالية.